In [1]:
#%%     Imports

%load_ext autoreload
%autoreload 2
%matplotlib inline

# Standard Imports
import tensorflow as tf
import keras_tuner as kt 
import kormos

tf.keras.backend.clear_session()
tf.keras.backend.set_floatx('float64')
tf_float = 'float64'
tf.random.set_seed(42)

import numpy as np
np_float = np.float64

import os
import datetime

# Own imports
import ContinuumMechanics as CM
import layers
import subANNs
import Outputs
import Plots
import Callbacks
import utils
import fit
import build


device = 'gpu:' + str(0) if tf.test.is_gpu_available() else 'cpu:0'

Instructions for updating:
Use `tf.config.list_physical_devices('GPU')` instead.


In [ ]:
dataset = 'isotropic_multistep'
load_model = True
modelToLoad = '.\\Results_{:}\\20251218-200838'.format(dataset)

pathToData = '.\\datasets\\{:}_data\\'.format(dataset)   

if load_model: 
    outputFolder = modelToLoad
else:
    date = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    outputFolder = '.\\Results_{:}\\'.format(dataset) + date

if not os.path.exists(outputFolder):
    os.makedirs(outputFolder)


In [ ]:


incomp = True
visco = True

uncoupled = True # must be True, was only experementally tested
rateDependent = False # if relaxation times are strain rate dependent

numDir = 0 # number $n$ of preferred material directions $\vec{m}$ (e.g. fibers) to be learned by the model
numTens = 1  # number $R$ of generalized Maxwell models $\tilde{\tns{L}}_r$ to be learned by the model
numExtra = 0 # number $n_f$ of features which parameterize the free energy functions and relaxation times
numExtraStruc = 0 # number of features which parameterize the preferred material direction $\vec{m}_i$ and corresponding wieghts w_i^{(r)}

# for saving and loading    
custom_objects = {
    'dirModelOrtho': CM.dirModelOrtho,
    'weightModelOrtho': CM.weightModelOrtho,
    'PositiveHeNormal': subANNs.PositiveHeNormal,
    'PositiveVarianceScaling': subANNs.PositiveVarianceScaling,
    'PositiveGlorotNormal': subANNs.PositiveGlorotNormal,
    'PsiSigmaLayer': CM.PsiSigmaLayer, 
    'GradientLayer': CM.GradientLayer,
    'ScaleLayer': layers.ScaleLayer, 
    'stressUpdateLayer': CM.stressUpdateLayer,
    'SparsityRegularizer': utils.SparsityRegularizer
}

In [ ]:
#%% Hyperparameters
#
lr = 0.01
clipnorm = None
lambda_maxwell = 0.0 # penalty to enforce sparsity of the Prony Series

# Time scaling constants (powers of 10)
tau_min = 0
tau_max = 2

# Some activation functions
acti0 = 'tanh'
acti1 = 'sigmoid'
acti2 = 'softplus'
acti3 = 'linear'
acti4 = 'elu'
acti5 = 'squared_softplus'
acti6 = 'neg_squared_softplus'

#
EPOCHS = 10000
earlyStopPatience = 10000

# maximum number N_max of Maxwell elements per generalized Maxwell model
nMaxwell = 3

# Equilibrium free energy: number of layers / neurons per layer
layer_size_psi = [10,] # last layer of shape (1,) is automatically included in the model
activations_psi = [acti2, acti2, acti2]

# Preferred material directions: number of layers / neurons per layer
layer_size_dir = [5,]
activations_dir = [acti2, acti3, acti3]

# Weights of preferred material directions: number of layers / neurons per layer
layer_size_dir = [5,] 
activations_w = [acti2, acti3, acti2]

# Relaxation times: : number of layers / neurons per layer
layer_size_tau = [10,] # last layer of shape (1,) is automatically included in the model
activations_tau = [acti2, acti2, acti2, acti2]

# Non-equilibrium free energy: number of layers / neurons per layer
layer_size_psi_a = [10,] # last layer of shape (1,) is automatically included in the model
activations_psi_a = [acti2, acti2, acti2, acti2]

In [5]:
#%% Data
        
# Training and validation data
trainDs = tf.data.experimental.load(pathToData + "ds_train_defGrad", compression='GZIP')
valDs = tf.data.experimental.load(pathToData + "ds_train_defGrad", compression='GZIP')

tf.data.experimental.save(trainDs, outputFolder + '\\ds_train_defGrad', compression='GZIP')
tf.data.experimental.save(valDs, outputFolder + '\\ds_valid_defGrad', compression='GZIP')

trainDs_infy= tf.data.experimental.load(pathToData + "ds_train_eq_defGrad", compression='GZIP')

# instDs = tf.data.experimental.load(pathToData + "ds_inst_ref_defGrad", compression='GZIP')
# ds_step = tf.data.experimental.load(pathToData + "ds_step_defGrad", compression='GZIP')
# ds_step_ref = tf.data.experimental.load(pathToData + "ds_step_ref_defGrad", compression='GZIP')

Instructions for updating:
Use `tf.data.Dataset.load(...)` instead.
Instructions for updating:
Use `tf.data.Dataset.save(...)` instead.


In [16]:
#%% build the model with prescribed hyperparameters
nSteps = int(np.loadtxt(pathToData + 'n_time_steps.txt'))

if load_model == True:
    model_fit  = Outputs.loadModel(modelToLoad, 'model_31', custom_objects)
    
else:             
    model_fit, model_full = build.build_model(nSteps,
                                                  numTens,
                                                  numDir,
                                                  nMaxwell,
                                                  numExtra,
                                                  numExtraStruc,
                                                  uncoupled,
                                                  rateDependent,
                                                  layer_size_psi,
                                                  activations_psi,
                                                  layer_size_dir,
                                                  activations_dir,
                                                  layer_size_w,
                                                  activations_w,
                                                  layer_size_tau,
                                                  activations_tau,
                                                  layer_size_psi_a, 
                                                  activations_psi_a,
                                                  [tau_min, tau_max],
                                                  lambda_maxwell,
                                                  incomp,
                                                  visco,
                                                  tf_float
                                                )
        
    

In [21]:
model_fit.load_weights(outputFolder + '\\ckpt\\ckpt-epoch-1881.ckpt')

In [123]:
# instantaneous elastic mode
S_infy  = model_full.get_layer('S_infy').output
F    = model_full.get_layer('F_input').input
time = model_full.get_layer('time_input').input
P_infy  = tf.keras.layers.Lambda(lambda x:  tf.matmul(x[0], x[1]))([F, S_infy])
model_full_infy = tf.keras.models.Model([F,], P_infy)


# history, model_full_infy = fit.deterministic(model_full_infy, trainDs_infy, 500, 500, outputFolder, valDs=trainDs_infy, Maxwell_monitor=None, loss=tf.keras.losses.MeanSquaredError())
model_full_infy.trainable = False
Plots.plot_stress(model_full_infy, trainDs_infy, trainDs_infy, outputFolder)
Plots.plot_elastic_stress_biaxial(model_full_infy, trainDs_infy, trainDs_infy, outputFolder)

1/1 [==============================] - 0s 94ms/step


In [97]:
model_full_infy.trainable = False


In [117]:
#%% Fit
normalized_error = False
stochastic = False

Maxwell_monitor = None # Callbacks.NonZeroWeightsMonitor(numTens, lambda_prony)
reg_callback = Callbacks.RegularizationCallback(lambda_maxwell, numTens)

if normalized_error == True:
    loss = fit.IndividuallyNormalizedLoss()  
else:
    # loss = tf.keras.losses.MeanSquaredError()
    loss = fit.WeightedLoss()


# stochastic optimizer
if stochastic:

    optimizer = tf.keras.optimizers.Adam(learning_rate=lr, clipnorm=clipnorm, name='optimizer')
    
    #Outputs.saveOptimizerConfig(optimizer, outputFolder=outputFolder)
    #Outputs.saveModelConfig(model_fit, outputFolder=outputFolder) 
    
    # tf.keras.backend.set_value(model_fit.optimizer.learning_rate, 0.1)
    model_fit.compile(
            optimizer=optimizer,
            loss=loss,
            run_eagerly=False
        )
    
    with tf.device('/device:CPU:0'):
        history, model_fit =  fit.stochastic(model_fit, trainDs, EPOCHS, earlyStopPatience, outputFolder, valDs=valDs, Maxwell_monitor=Maxwell_monitor)
                                    
# deterministic optimizer
else:
    with tf.device('/device:CPU:0'):
        history, model_fit = fit.deterministic(model_fit, trainDs, EPOCHS, earlyStopPatience, outputFolder, valDs=valDs, Maxwell_monitor=Maxwell_monitor, loss=loss)


#%%    
Outputs.saveLoss(history, Maxwell_monitor=Maxwell_monitor, outputFolder=outputFolder)
Outputs.plotLoss(history, Maxwell_monitor=Maxwell_monitor, title='$\Lambda = {:}$'.format(lambda_maxwell), outputFolder=outputFolder,scale='log')

Constrained model: 

   model_psi_a_0
      Psi_a_1_1_1:
         kernel constrained
         no bias constraint
      Psi_a_1_2_1:
         kernel constrained
         no bias constraint
      Psi_a_1_3_1:
         kernel constrained
         no bias constraint
      Psi_a_1_1_2:
         kernel constrained
      Psi_a_1_2_2:
         kernel constrained
      Psi_a_1_3_2:
         kernel constrained
   model_Psi
   model_tau_0
      Tau_1_1_1:
         no kernel constraint
         no bias constraint
      Tau_1_2_1:
         no kernel constraint
         no bias constraint
      Tau_1_3_1:
         no kernel constraint
         no bias constraint
      Tau_1_1_2:
         no kernel constraint
         no bias constraint
      Tau_1_2_2:
         no kernel constraint
         no bias constraint
      Tau_1_3_2:
         no kernel constraint
         no bias constraint
Total number bounds set      :  243

Epoch 21: saving model to C:\Users\Kian\Documents\Projekte\ViscoCANN\vCANN\Result

In [19]:
model_fit.load_weights(outputFolder + '\\ckpt\\ckpt-epoch-4061.ckpt')

((array([[[[1.        , 0.        , 0.        ],
           [0.        , 1.        , 0.        ],
           [0.        , 0.        , 1.        ]],
  
          [[1.0020577 , 0.        , 0.        ],
           [0.        , 0.9989728 , 0.        ],
           [0.        , 0.        , 0.9989728 ]],
  
          [[1.0082304 , 0.        , 0.        ],
           [0.        , 0.99591   , 0.        ],
           [0.        , 0.        , 0.99591   ]],
  
          ...,
  
          [[2.5       , 0.        , 0.        ],
           [0.        , 0.6324555 , 0.        ],
           [0.        , 0.        , 0.6324555 ]],
  
          [[2.5       , 0.        , 0.        ],
           [0.        , 0.6324555 , 0.        ],
           [0.        , 0.        , 0.6324555 ]],
  
          [[2.5       , 0.        , 0.        ],
           [0.        , 0.6324555 , 0.        ],
           [0.        , 0.        , 0.6324555 ]]],
  
  
         [[[1.        , 0.        , 0.        ],
           [0.        ,

In [71]:
model_fit.weights

[<tf.Variable 'Psi_a_1_1_1/kernel:0' shape=(2, 10) dtype=float64, numpy=
 array([[ 0.79317156,  0.36385094, -1.42851162, -1.4134486 , -1.20223614,
          0.96575842,  0.31447555,  0.25228279, -1.83192429,  0.23634304],
        [ 0.33891919,  0.70849486,  2.12801135, -0.7324365 ,  0.97113399,
         -0.39193984,  0.81181634,  0.26105921, -1.83442715, -0.6027282 ]])>,
 <tf.Variable 'Psi_a_1_1_1/bias:0' shape=(10,) dtype=float64, numpy=
 array([ 0.32941801, -0.02591699,  0.50055013, -0.46446895,  0.38299524,
        -0.05148593,  0.17759281,  0.26084261, -0.57735947, -0.13833821])>,
 <tf.Variable 'Psi_a_1_2_1/kernel:0' shape=(2, 10) dtype=float64, numpy=
 array([[-0.83172325, -0.48653132,  0.05773602, -0.79589916, -1.37626455,
         -0.63973492, -0.37822157,  1.3027768 ,  1.06585166,  1.48074709],
        [ 4.26260469, -0.91815047,  1.84167785, -1.01732235, -0.78850841,
         -1.24202765,  1.08184443, -0.1333544 ,  0.52078775, -0.78266615]])>,
 <tf.Variable 'Psi_a_1_2_1/bias:0'

In [ ]:
#%% Plot graphs and save model (summary)

Outputs.showModelSummary(model_fit, outputFolder)
Outputs.plotModelGraph(model_fit, outputFolder)
Outputs.saveModel(model_fit, outputFolder)

from exportArchitecture import exportArchitecture
# exportArchitecture(model_fit, outputFolder, weights_local=False)
exportArchitecture(model_fit, outputFolder, weights_local=True)

#%%
# loadedModel = Outputs.loadModel(outputFolder, 'model', custom_objects=custom_objects)





Model: "model_31"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 F_input (InputLayer)           [(None, 1000, 3, 3)  0           []                               
                                ]                                                                 
                                                                                                  
 tf.compat.v1.shape_5 (TFOpLamb  (4,)                0           ['F_input[0][0]']                
 da)                                                                                              
                                                                                                  
 tf.__operators__.getitem_180 (  ()                  0           ['tf.compat.v1.shape_5[0][0]']   
 SlicingOpLambda)                                                                      

In [32]:
### uniaxial tension
stretch_max = 2.5
stretch_min = 1
lam_dot = 0.1
lam = np.linspace(stretch_min,stretch_max,nSteps)

F = np.zeros([1, len(lam), 3, 3])
F[:,:,0,0] = lam
F[:,:,1,1] = 1.0/np.sqrt(lam)
F[:,:,2,2] = 1.0/np.sqrt(lam)

total_time = (stretch_max-stretch_min)/lam_dot*2
time = np.linspace(0,total_time,len(lam)+1)[:-1].reshape(1,-1)

inp=[F,time]
res = model_full.predict(inp)

invars = res[0]
Psi_a = res[-11]
Psi = res[1]

import matplotlib.pyplot as plt
fig, ax = plt.subplots()
ax.plot(lam, Psi[0,:,0], color='k', label='$\Psi_\infty$')

ax.plot(lam, Psi_a[0,:,0], label='$\Psi_{a1}$')
ax.plot(lam, Psi_a[0,:,1], label='$\Psi_{a2}$')
ax.plot(lam, Psi_a[0,:,2], label='$\Psi_{a3}$')

ax.plot(lam, 1.54*Psi[0,:,0], color='tab:blue', linestyle='--', label='$\Psi_{a1}^{ref}$')
ax.plot(lam, 1.31*Psi[0,:,0], color='tab:orange', linestyle='--', label='$\Psi_{a2}^{ref}$')
ax.plot(lam, 0.14*Psi[0,:,0], color='tab:green', linestyle='--', label='$\Psi_{a3}^{ref}$')

ax.grid()
ax.legend()

### Equibiaxial tension
stretch_max = 1.3
stretch_min = 1
lam_dot = 0.1
lam = np.linspace(stretch_min,stretch_max,nSteps)

F = np.zeros([1, len(lam), 3, 3])
F[:,:,0,0] = lam
F[:,:,1,1] = lam
F[:,:,2,2] = 1.0/lam**2

total_time = (stretch_max-stretch_min)/lam_dot*2
time = np.linspace(0,total_time,len(lam)+1)[:-1].reshape(1,-1)

inp=[F,time]
res = model_full.predict(inp)

invars = res[0]
Psi_a = res[-11]
Psi = res[1]

import matplotlib.pyplot as plt
fig, ax = plt.subplots()
ax.plot(lam, Psi[0,:,0], color='k', label='$\Psi_\infty$')

ax.plot(lam, Psi_a[0,:,0], label='$\Psi_{a1}$')
ax.plot(lam, Psi_a[0,:,1], label='$\Psi_{a2}$')
ax.plot(lam, Psi_a[0,:,2], label='$\Psi_{a3}$')

ax.plot(lam, 1.54*Psi[0,:,0], color='tab:blue', linestyle='--', label='$\Psi_{a1}^{ref}$')
ax.plot(lam, 1.31*Psi[0,:,0], color='tab:orange', linestyle='--', label='$\Psi_{a2}^{ref}$')
ax.plot(lam, 0.14*Psi[0,:,0], color='tab:green', linestyle='--', label='$\Psi_{a3}^{ref}$')

ax.grid()
ax.legend()

### pure shear
stretch_max = 1.6
stretch_min = 1
lam_dot = 0.1
lam = np.linspace(stretch_min,stretch_max,nSteps)


F = np.zeros([1, len(lam), 3, 3])
F[:,:,0,0] = lam
F[:,:,1,1] = 1.0
F[:,:,2,2] = 1.0/lam

total_time = (stretch_max-stretch_min)/lam_dot*2
time = np.linspace(0,total_time,len(lam)+1)[:-1].reshape(1,-1)

inp=[F,time]
res = model_full.predict(inp)

invars = res[0]
Psi_a = res[-11]
Psi = res[1]

import matplotlib.pyplot as plt
fig, ax = plt.subplots()
ax.plot(lam, Psi[0,:,0], color='k', label='$\Psi_\infty$')

ax.plot(lam, Psi_a[0,:,0], label='$\Psi_{a1}$')
ax.plot(lam, Psi_a[0,:,1], label='$\Psi_{a2}$')
ax.plot(lam, Psi_a[0,:,2], label='$\Psi_{a3}$')

ax.plot(lam, 1.54*Psi[0,:,0], color='tab:blue', linestyle='--', label='$\Psi_{a1}^{ref}$')
ax.plot(lam, 1.31*Psi[0,:,0], color='tab:orange', linestyle='--', label='$\Psi_{a2}^{ref}$')
ax.plot(lam, 0.14*Psi[0,:,0], color='tab:green', linestyle='--', label='$\Psi_{a3}^{ref}$')

ax.grid()
ax.legend()

1/1 [==============================] - 0s 92ms/step


In [120]:
n = 1
F[0,n]


S_infty = res[16]
Tau = res[17]
S = res[18][0,n] 
# S_neq = res[39]
dPsi_a_dI_ref = res[25][1]
dPsi_a_dJ_ref = res[26][2]
alpha_a = res[28]
beta_a = res[29]
S_a = res[32]
Q_unstack = res[33]
P = res[34]
# dPsi_a_dI_c = res[-9]
# dPsi_a_dJ_c = res[-8]
# ddPsi_a_ddI_c = res[-7]
# ddPsi_a_ddJ_c = res[-6]
# ddPsi_a_dIdJ_c = res[-5]
# S_ra_neq_bar = res[38]
# dSdCbar = res[35]
# dS_a_dC_bar = res[36]
# dS_a_neq_bar_1, dS_a_neq_bar_2, dS_a_neq_bar_3 = res[-3], res[-2], res[-1]

# Q_unstack[2][0,]
# time[0,n]
#S_a[0,n]
#Tau[0,n]
# dPsi_a_dI_c[0,n]
# print(S_ra_neq_bar[0,n,0])
# print(dS_a_neq_bar_1[n][0,n])
# print(dS_a_neq_bar_2[n][0,n])
# print(dS_a_neq_bar_3[n][0,n])
# Tau[0,n]
# print((dS_a_dC_bar[0][0,n,0,0,:,:] + dS_a_dC_bar[0][0,n,0,0,:,:] + dS_a_dC_bar[0][0,n,0,0,:,:])*2)

# # dQdCbar_2[0][0,0]
# print(dS_a_neq_bar_3[n][0,n,2,2])





In [121]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots()
ax.plot(lam,Tau[0,:,0], color='tab:blue', label='$\\tau_1$')
ax.plot(lam,Tau[0,:,1], color='tab:orange', label='$\\tau_2$')
ax.plot(lam,Tau[0,:,2], color='tab:green', label='$\\tau_3$')

ax.plot(lam, np.ones_like(lam) * 1.00, color='tab:blue', linestyle='--', label='$\\tau_1^{ref}$')
ax.plot(lam, np.ones_like(lam) * 15.98, color='tab:orange', linestyle='--', label='$\\tau_2^{ref}$')
ax.plot(lam, np.ones_like(lam) * 50.01, color='tab:green', linestyle='--', label='$\\tau_3^{ref}$')

ax.set_yscale('log')
ax.grid(True)
ax.legend()


In [22]:
# ckpt = "C:\\Users\\Kian\\Documents\\Projekte\\ViscoCANN\\vCANN\\Results_Hossain\\20251030-185939\\ckpt\\ckpt-epoch-1001.ckpt"
# ckpt = "C:\\Users\\Kian\\Documents\\Projekte\\ViscoCANN\\vCANN\\Results_Hossain\\20251103-190018\\ckpt\\ckpt-epoch-561.ckpt"
# loaded_model = Outputs.loadModel(outputFolder, 'model', custom_objects)
# model_fit.load_weights(ckpt)
#%% individial plotting
    
%matplotlib qt
#%% Structural Tensor
# Plots.plot_struc_tensor(model_fit, plotly=False, outputFolder=outputFolder)

#%% Stress Response 
Plots.plot_multi_step_isotropic(model_fit, trainDs, outputFolder)
# Plots.plot_stress(model_fit, trainDs, valDs, outputFolder)
# Plots.plot_multi_step(model_fit, trainDs, outputFolder)
# Plots.plot_stress(model_fit, ds_step, ds_step_ref, outputFolder)

        

1/1 [==============================] - 0s 145ms/step


In [ ]:
from checkpoint_analysis import parallel_checkpoint_analysis
parallel_checkpoint_analysis(model_fit, pathToData, outputFolder, rateDependent, checkpoint_dir=None, max_workers=None)

In [ ]:
from checkpoint_analysis import create_video_from_checkpoint_analysis
video_folder = "C:\\Users\\Kian\\Documents\\Projekte\\ViscoCANN\\vCANN\\Results_General_biaxial\\20251125-015147"
create_video_from_checkpoint_analysis(video_folder, fps=2, duration_per_frame=0.5)